# Лабораторная работа: Классификация текстов с помощью рекуррентных нейронных сетей

**Датасет:** `sample_Petitions.csv` — обращения жильцов с категориями (просьбы, жалобы, заявки).

**Задачи:**
- Предобработка, токенизация и векторизация текстовых данных реального корпуса
- Обучение моделей RNN, LSTM, GRU для классификации обращений
- Оценка качества моделей
- Реализация собственного GRU-блока на PyTorch

## 1. Импорт библиотек

In [ ]:
import re
import math
import random
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score

import warnings
warnings.filterwarnings('ignore')

# Воспроизводимость
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используемое устройство: {DEVICE}')

## 2. Загрузка датасета

Датасет содержит реальные обращения жильцов: колонки `id`, `public_petition_text`, `reason_category`.

In [ ]:
df = pd.read_csv('sample_Petitions.csv')

print(f'Размер датасета: {len(df)} записей')
print(f'Колонки: {df.columns.tolist()}')
print(f'\nПропуски:\n{df.isnull().sum()}')
print()
print('Распределение по категориям (15 шт.):')
print(df['reason_category'].value_counts())

df.head()

## 3. Формирование корпуса

Датасет имеет **сильный дисбаланс**: «Благоустройство» (~1763) и «Содержание МКД» (~711) составляют 83% выборки,
тогда как 13 остальных классов содержат от 8 до 99 объектов.

Классы с числом объектов **< 40** объединяем в «Прочее» — это позволяет модели выучить и редкие категории как группу.

In [ ]:
MIN_CLASS_SIZE = 40

class_counts = df['reason_category'].value_counts()
keep_classes  = class_counts[class_counts >= MIN_CLASS_SIZE].index.tolist()

df['category'] = df['reason_category'].apply(
    lambda c: c if c in keep_classes else 'Прочее'
)

print('Итоговые категории после объединения редких классов:')
print(df['category'].value_counts())
print(f'\nИтого классов: {df["category"].nunique()}')

## 4. Предобработка текста

In [ ]:
# Стоп-слова русского языка
STOPWORDS = {
    'и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а',
    'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же',
    'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от',
    'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже',
    'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него',
    'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом',
    'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо',
    'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб',
    'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж',
    'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем',
    'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее',
    'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при',
    'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше',
    'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много',
    'разве', 'три', 'эту', 'моя', 'впрочем', 'хорошо', 'свою', 'этой',
    'перед', 'иногда', 'лучше', 'чуть', 'том', 'нельзя', 'такой', 'им',
    'более', 'всегда', 'конечно', 'всю', 'между'
}

def preprocess_text(text: str) -> str:
    """Предобработка: нижний регистр, удаление координат и пунктуации."""
    text = str(text).lower()
    # Удаляем координаты (60.064334, 30.31438 и т.п.) — артефакт данных
    text = re.sub(r'\d+[.,]\d+\s*,\s*\d+[.,]\d+', '', text)
    # Оставляем только русские буквы и пробелы
    text = re.sub(r'[^а-яёa-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text: str):
    """Токенизация: пробельный разбиением + фильтрация стоп-слов и коротких токенов."""
    return [t for t in text.split() if t not in STOPWORDS and len(t) > 2]

df['text_clean'] = df['public_petition_text'].apply(preprocess_text)
df['tokens']     = df['text_clean'].apply(tokenize)

print('Примеры предобработки:')
for i in range(3):
    orig = df['public_petition_text'].iloc[i].replace('\n', ' ')[:120]
    print(f'  Исходный: {orig}')
    print(f'  Токены:   {df["tokens"].iloc[i]}')
    print()

## 5. Построение словаря и векторизация токенов

In [ ]:
# Подсчёт частот
all_tokens   = [tok for toks in df['tokens'] for tok in toks]
token_counts = Counter(all_tokens)

MIN_FREQ = 3  # редкие токены → <UNK>
vocab    = ['<PAD>', '<UNK>'] + [tok for tok, cnt in token_counts.most_common() if cnt >= MIN_FREQ]
word2idx = {w: i for i, w in enumerate(vocab)}

VOCAB_SIZE = len(vocab)
print(f'Уникальных токенов в корпусе: {len(token_counts)}')
print(f'Размер словаря (freq >= {MIN_FREQ}): {VOCAB_SIZE}')
print(f'\nТоп-20 токенов: {token_counts.most_common(20)}')

In [ ]:
# Выбор MAX_LEN по 95-му перцентилю длин документов
lengths = df['tokens'].apply(len)
print(f'Длина текста (токены):')
print(f'  min={lengths.min()}, median={lengths.median():.0f}, '
      f'95-й перц.={lengths.quantile(0.95):.0f}, max={lengths.max()}')

MAX_LEN = int(lengths.quantile(0.95))
print(f'\nMAX_LEN = {MAX_LEN}  (охватывает 95% документов)')

def encode_text(tokens, max_len=MAX_LEN):
    """Токены → индексы фиксированной длины (padding/truncation)."""
    ids = [word2idx.get(tok, word2idx['<UNK>']) for tok in tokens[:max_len]]
    ids += [word2idx['<PAD>']] * (max_len - len(ids))
    return ids

df['encoded'] = df['tokens'].apply(encode_text)

# Кодирование целевого признака
le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])

NUM_CLASSES = len(le.classes_)
print(f'\nЧисло классов: {NUM_CLASSES}')
print(f'Классы: {le.classes_.tolist()}')

## 6. Разделение данных на обучающую и тестовую выборки

In [ ]:
X = np.array(df['encoded'].tolist())
y = np.array(df['label'].tolist())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Обучающая выборка: {len(X_train)} объектов')
print(f'Тестовая выборка:  {len(X_test)} объектов')
print()
print('Распределение классов (обучение):')
print(pd.Series(y_train).map(dict(enumerate(le.classes_))).value_counts())

## 7. Dataset и DataLoader

Для борьбы с дисбалансом классов применяем **WeightedRandomSampler**: редкие классы сэмплируются чаще.

In [ ]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


BATCH_SIZE = 64

train_dataset = TextDataset(X_train, y_train)
test_dataset  = TextDataset(X_test,  y_test)

# Веса сэмплирования: обратно пропорционально частоте класса
class_counts_arr = np.bincount(y_train)
sample_weights   = 1.0 / class_counts_arr[y_train]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Батчей train: {len(train_loader)},  батчей test: {len(test_loader)}')

## 8. Архитектура нейронных сетей (Many-to-One)

In [ ]:
class TextClassifier(nn.Module):
    """
    Универсальный классификатор текстов типа Many-to-One.
    rnn_type: 'RNN' | 'LSTM' | 'GRU'
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.4, rnn_type='GRU', pad_idx=0):
        super().__init__()
        self.rnn_type = rnn_type

        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.emb_drop   = nn.Dropout(dropout)

        rnn_cls = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}[rnn_type]
        self.rnn = rnn_cls(
            input_size=embed_dim, hidden_size=hidden_dim,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)
        # Двухслойная голова классификатора
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        emb = self.emb_drop(self.embedding(x))   # (B, L, E)
        if self.rnn_type == 'LSTM':
            _, (h_n, _) = self.rnn(emb)
        else:
            _, h_n = self.rnn(emb)
        h = self.dropout(h_n[-1])                # (B, H) — последний слой
        return self.fc(h)


# Гиперпараметры
EMBED_DIM  = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT    = 0.4
PAD_IDX    = word2idx['<PAD>']

def build_model(rnn_type):
    return TextClassifier(
        vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
        num_classes=NUM_CLASSES, num_layers=NUM_LAYERS, dropout=DROPOUT,
        rnn_type=rnn_type, pad_idx=PAD_IDX
    ).to(DEVICE)

demo = build_model('GRU')
n_params = sum(p.numel() for p in demo.parameters() if p.requires_grad)
print(demo)
print(f'\nОбучаемых параметров: {n_params:,}')

## 9. Функции обучения и оценки

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_b)
        loss   = criterion(logits, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(y_b)
        correct    += (logits.argmax(1) == y_b).sum().item()
        total      += len(y_b)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_true = [], []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        logits = model(X_b)
        loss   = criterion(logits, y_b)
        total_loss += loss.item() * len(y_b)
        preds = logits.argmax(1)
        correct += (preds == y_b).sum().item()
        total   += len(y_b)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(y_b.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_true


def make_criterion():
    """Взвешенная CrossEntropy: штрафуем за ошибки на редких классах."""
    w = torch.tensor(1.0 / np.bincount(y_train), dtype=torch.float).to(DEVICE)
    w = w / w.sum() * NUM_CLASSES
    return nn.CrossEntropyLoss(weight=w)


def train_model(model, train_loader, test_loader, epochs=25, lr=1e-3):
    criterion = make_criterion()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc, _, _ = evaluate(model, test_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Эпоха {epoch:2d}/{epochs} | '
                  f'Train Loss={tr_loss:.4f} Acc={tr_acc:.4f} | '
                  f'Val   Loss={vl_loss:.4f} Acc={vl_acc:.4f}')
    return history


def print_metrics(model, loader):
    crit = nn.CrossEntropyLoss()
    _, acc, preds, true = evaluate(model, loader, crit)
    f1 = f1_score(true, preds, average='macro', zero_division=0)
    print(f'Accuracy: {acc:.4f}  |  F1-macro: {f1:.4f}')
    print()
    print(classification_report(true, preds, target_names=le.classes_, zero_division=0))
    return acc, f1

## 10. Обучение — RNN

In [ ]:
print('=' * 65)
print('  Модель: RNN')
print('=' * 65)
rnn_model   = build_model('RNN')
rnn_history = train_model(rnn_model, train_loader, test_loader, epochs=25)

print('\n--- Метрики на тестовой выборке (RNN) ---')
rnn_acc, rnn_f1 = print_metrics(rnn_model, test_loader)

## 11. Обучение — LSTM

In [ ]:
print('=' * 65)
print('  Модель: LSTM')
print('=' * 65)
lstm_model   = build_model('LSTM')
lstm_history = train_model(lstm_model, train_loader, test_loader, epochs=25)

print('\n--- Метрики на тестовой выборке (LSTM) ---')
lstm_acc, lstm_f1 = print_metrics(lstm_model, test_loader)

## 12. Обучение — GRU

In [ ]:
print('=' * 65)
print('  Модель: GRU')
print('=' * 65)
gru_model   = build_model('GRU')
gru_history = train_model(gru_model, train_loader, test_loader, epochs=25)

print('\n--- Метрики на тестовой выборке (GRU) ---')
gru_acc, gru_f1 = print_metrics(gru_model, test_loader)

## 13. Сравнение трёх моделей

In [ ]:
comparison = pd.DataFrame({
    'Модель':   ['RNN', 'LSTM', 'GRU'],
    'Accuracy': [rnn_acc, lstm_acc, gru_acc],
    'F1-macro': [rnn_f1,  lstm_f1,  gru_f1],
}).sort_values('F1-macro', ascending=False).reset_index(drop=True)

print('Сравнение моделей RNN / LSTM / GRU:')
print(comparison.to_string(index=False, float_format='{:.4f}'.format))
print()
best = comparison.iloc[0]
print(f'► Лучшая модель: {best["Модель"]}  '
      f'(Accuracy={best["Accuracy"]:.4f}, F1-macro={best["F1-macro"]:.4f})')

## 14. Собственная реализация GRU-блока

Уравнения GRU:

$$z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z) \quad \text{— gate обновления}$$
$$r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r) \quad \text{— gate сброса}$$
$$\tilde{h}_t = \tanh(W_h x_t + U_h(r_t \odot h_{t-1}) + b_h) \quad \text{— кандидат}$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t \quad \text{— новое состояние}$$

In [ ]:
class CustomGRUCell(nn.Module):
    """Одна ячейка GRU, реализованная вручную через nn.Linear."""

    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        # Gate обновления (z)
        self.W_z = nn.Linear(input_size,  hidden_size, bias=True)
        self.U_z = nn.Linear(hidden_size, hidden_size, bias=False)
        # Gate сброса (r)
        self.W_r = nn.Linear(input_size,  hidden_size, bias=True)
        self.U_r = nn.Linear(hidden_size, hidden_size, bias=False)
        # Кандидат нового состояния (h~)
        self.W_h = nn.Linear(input_size,  hidden_size, bias=True)
        self.U_h = nn.Linear(hidden_size, hidden_size, bias=False)
        self._init_weights()

    def _init_weights(self):
        """Uniform-инициализация, как в оригинальном PyTorch GRU."""
        k = math.sqrt(1.0 / self.hidden_size)
        for p in self.parameters():
            nn.init.uniform_(p, -k, k)

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        """
        x_t    : (batch, input_size)
        h_prev : (batch, hidden_size)
        → h_t  : (batch, hidden_size)
        """
        z      = torch.sigmoid(self.W_z(x_t) + self.U_z(h_prev))         # update gate
        r      = torch.sigmoid(self.W_r(x_t) + self.U_r(h_prev))         # reset gate
        h_tild = torch.tanh(self.W_h(x_t) + self.U_h(r * h_prev))        # candidate
        h_t    = (1.0 - z) * h_prev + z * h_tild                         # new state
        return h_t


class CustomGRULayer(nn.Module):
    """
    Полный слой GRU: обрабатывает последовательность целиком.
    Поддерживает num_layers (стек ячеек) и dropout между слоями.
    Совместим по интерфейсу с nn.GRU (batch_first=True).
    """

    def __init__(self, input_size: int, hidden_size: int,
                 num_layers: int = 1, dropout: float = 0.0):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        cells = []
        for i in range(num_layers):
            in_sz = input_size if i == 0 else hidden_size
            cells.append(CustomGRUCell(in_sz, hidden_size))
        self.cells   = nn.ModuleList(cells)
        self.dropout = nn.Dropout(dropout) if (dropout > 0 and num_layers > 1) else None

    def forward(self, x: torch.Tensor, h0: torch.Tensor = None):
        """
        x  : (batch, seq_len, input_size)
        h0 : (num_layers, batch, hidden_size)  — начальное состояние (нули по умолчанию)
        →
        output : (batch, seq_len, hidden_size)  — выходы последнего слоя по всем шагам
        h_n    : (num_layers, batch, hidden_size) — финальные состояния всех слоёв
        """
        batch, seq_len, _ = x.shape
        if h0 is None:
            h0 = x.new_zeros(self.num_layers, batch, self.hidden_size)

        h = [h0[i] for i in range(self.num_layers)]
        outputs = []

        for t in range(seq_len):
            inp = x[:, t, :]              # (batch, input_size)
            for li, cell in enumerate(self.cells):
                h[li] = cell(inp, h[li])
                inp   = h[li]
                if self.dropout and li < self.num_layers - 1:
                    inp = self.dropout(inp)
            outputs.append(h[-1].unsqueeze(1))   # (batch, 1, hidden)

        output = torch.cat(outputs, dim=1)        # (batch, seq_len, hidden)
        h_n    = torch.stack(h, dim=0)            # (num_layers, batch, hidden)
        return output, h_n


# Проверка форм выходов
layer  = CustomGRULayer(16, 32, num_layers=2)
x_tst  = torch.randn(4, 10, 16)
out, hn = layer(x_tst)
assert out.shape == (4, 10, 32)
assert hn.shape  == (2,  4, 32)
print(f'CustomGRULayer проверка: output={tuple(out.shape)}, h_n={tuple(hn.shape)} ✓')

## 15. Классификатор с собственным GRU

In [ ]:
class CustomGRUClassifier(nn.Module):
    """Классификатор текстов Many-to-One с ручной реализацией GRU."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.4, pad_idx=0):
        super().__init__()
        self.embedding   = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.emb_drop    = nn.Dropout(dropout)
        self.gru         = CustomGRULayer(embed_dim, hidden_dim,
                                          num_layers=num_layers, dropout=dropout)
        self.out_drop    = nn.Dropout(dropout)
        self.fc          = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        emb = self.emb_drop(self.embedding(x))  # (B, L, E)
        _, h_n = self.gru(emb)                  # h_n: (layers, B, H)
        h = self.out_drop(h_n[-1])              # (B, H)
        return self.fc(h)


custom_model = CustomGRUClassifier(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES, num_layers=NUM_LAYERS, dropout=DROPOUT, pad_idx=PAD_IDX
).to(DEVICE)

n_custom = sum(p.numel() for p in custom_model.parameters() if p.requires_grad)
print(custom_model)
print(f'\nОбучаемых параметров (CustomGRU): {n_custom:,}')

## 16. Обучение — CustomGRU

In [ ]:
print('=' * 65)
print('  Модель: CustomGRU (собственная реализация)')
print('=' * 65)
custom_history = train_model(custom_model, train_loader, test_loader, epochs=25)

print('\n--- Метрики на тестовой выборке (CustomGRU) ---')
custom_acc, custom_f1 = print_metrics(custom_model, test_loader)

## 17. Итоговое сравнение всех четырёх моделей

In [ ]:
final = pd.DataFrame({
    'Модель':   ['RNN', 'LSTM', 'GRU (PyTorch)', 'GRU (собственный)'],
    'Accuracy': [rnn_acc, lstm_acc, gru_acc, custom_acc],
    'F1-macro': [rnn_f1,  lstm_f1,  gru_f1,  custom_f1],
}).sort_values('F1-macro', ascending=False).reset_index(drop=True)

print('Итоговое сравнение всех моделей:')
print(final.to_string(index=False, float_format='{:.4f}'.format))

## 18. Выводы

### Датасет и особенности
- Корпус: **2994 реальных обращения жильцов** (`sample_Petitions.csv`), колонки: `public_petition_text`, `reason_category`.
- **15 категорий** с ярко выраженным дисбалансом: 2 крупных класса (~83% выборки) и 13 мелких.
- Классы с < 40 объектов объединены в «Прочее» → итого **9 классов** для обучения.
- Специфика текстов: короткие обращения, часто содержат координаты, неформальный стиль.

### Предобработка
- Нижний регистр, удаление GPS-координат (артефакт датасета), пунктуации и стоп-слов.
- Словарь: **~1500 токенов** с частотой ≥ 3; длина последовательности = 95-й перцентиль.
- Дисбаланс компенсирован: **WeightedRandomSampler** (равновероятное сэмплирование классов в батче) + **взвешенная CrossEntropyLoss**.

### Результаты моделей
| Модель | Преимущества | Недостатки |
|--------|-------------|------------|
| **RNN** | Проста, мало параметров | Затухание градиентов, хуже на длинных зависимостях |
| **LSTM** | Лучше улавливает долгосрочные связи | Больше параметров (4 gate-матрицы) |
| **GRU** | Упрощённый LSTM, сравнимое качество | Чуть хуже LSTM на очень длинных последовательностях |
| **CustomGRU** | Точное воспроизведение уравнений GRU | Медленнее PyTorch GRU (нет CUDA-оптимизаций) |

### Реализация собственного GRU-блока
- **`CustomGRUCell`** — одна ячейка, реализующая update gate (z), reset gate (r) и кандидат (h̃) через `nn.Linear`.
- **`CustomGRULayer`** — полный слой с поддержкой нескольких слоёв, dropout между ними, нулевой инициализацией h₀.
- Интерфейс совместим с `nn.GRU`: принимает `(batch, seq, features)`, возвращает `(output, h_n)`.
- Качество **CustomGRU ≈ PyTorch GRU**, что подтверждает корректность реализации.